# Part 1 — WSJ Headline Sentiment and S&P 500 Returns

**Course**: ECOM217 — LLM and Textual Analysis in Finance
**Hypothesis**: Aggregate WSJ headline sentiment predicts S&P 500 returns, with negative sentiment carrying asymmetric (stronger) predictive power.

**Train period**: 2016-01-01 to 2021-12-31
**Test period**: 2022-01-01 to 2023-12-31

This notebook follows the project specification end-to-end: NLP pipeline (Sections 0-8), signal construction and strategies (Sections 9-12), evaluation (Sections 13-19). All parameters are chosen by financial convention before looking at results — no in-sample optimisation.

## Section 0 — Setup and Imports

Pin all random seeds for reproducibility. Imports cover numerical work, NLP, modelling, and statistics.

In [ ]:
import os
import re
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

from scipy import stats
import statsmodels.api as sm

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Plot defaults
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# NLTK resources (idempotent)
for resource in ['stopwords', 'punkt', 'punkt_tab',
                 'averaged_perceptron_tagger',
                 'averaged_perceptron_tagger_eng',
                 'wordnet', 'omw-1.4']:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

print('Setup complete. Seed pinned to', SEED)

## Section 1 — Load Datasets

Three sources:
- **Headlines**: WSJ headlines 2016-2023, pre-labelled by FinBERT (positive / neutral / negative). FinBERT labels are used as ground truth for training the in-house classifiers.
- **SPX**: daily S&P 500 returns. `sprtrn` is the simple total return.
- **FF3**: Fama-French daily factors. `rf` is the risk-free rate (we need it for the backtest cash leg and the FF3 regression).

In [ ]:
DATA_DIR = '.'

headlines = pd.read_csv(
    os.path.join(DATA_DIR, 'wsj_finbert_labeled_all.csv'),
    parse_dates=['date']
)
spx = pd.read_csv(
    os.path.join(DATA_DIR, 'ProjectA_spx_index_daily_2023.csv'),
    parse_dates=['date']
)
ff3 = pd.read_csv(
    os.path.join(DATA_DIR, 'Shared_ff_factors_daily_2023.csv'),
    parse_dates=['date']
)

print('=== Headlines ===')
print('Shape:', headlines.shape)
print('Date range:', headlines['date'].min().date(), 'to', headlines['date'].max().date())
print('Sentiment counts:')
print(headlines['sentiment'].value_counts())

print('\n=== SPX ===')
print('Shape:', spx.shape)
print('Date range:', spx['date'].min().date(), 'to', spx['date'].max().date())

print('\n=== FF3 ===')
print('Shape:', ff3.shape)
print('Date range:', ff3['date'].min().date(), 'to', ff3['date'].max().date())
print('Columns:', list(ff3.columns))

In [ ]:
print(ff3[['mktrf', 'smb', 'hml', 'rf']].describe().round(6))


## Section 2 — Data Cleaning

Two duplicate filters:
1. **Exact duplicates** — same row in full (likely scrape artifacts).
2. **Same-date + same-headline** — keep only one copy per day even if other columns (e.g. company list) vary slightly.

We also drop rows with a missing headline.

In [ ]:
print('Initial rows:', len(headlines))

# Drop missing headlines
headlines = headlines.dropna(subset=['headline']).reset_index(drop=True)
print('After dropping missing headlines:', len(headlines))

# Drop exact duplicates
headlines = headlines.drop_duplicates().reset_index(drop=True)
print('After dropping exact duplicates:', len(headlines))

# Drop same-date + same-headline
headlines = headlines.drop_duplicates(
    subset=['date', 'headline']
).reset_index(drop=True)
print('After dropping same-date+same-headline duplicates:', len(headlines))

## Section 3 — Text Preprocessing

Pipeline applied to each headline:
1. Lowercase
2. Remove punctuation and numbers (keep letters and whitespace only)
3. Tokenise with NLTK
4. Remove English stopwords
5. POS-tag and lemmatise (nouns/verbs/adjectives/adverbs each get the right POS to the lemmatiser)

The result is a clean, space-separated string suitable for vectorisation.

In [ ]:
STOPWORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()


def _to_wn_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    if tag.startswith('V'):
        return wordnet.VERB
    if tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN  # default


def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t and t not in STOPWORDS and len(t) > 1]
    if not tokens:
        return ''
    tagged = pos_tag(tokens)
    lemmas = [LEMMATIZER.lemmatize(t, _to_wn_pos(p)) for t, p in tagged]
    return ' '.join(lemmas)


# Show 3 examples
sample_idx = [0, 100, 1000]
for i in sample_idx:
    raw = headlines.iloc[i]['headline']
    clean = preprocess(raw)
    print(f'BEFORE: {raw}')
    print(f'AFTER : {clean}')
    print('-' * 60)

In [ ]:
# Apply to all headlines (this takes a minute or two)
headlines['clean_text'] = headlines['headline'].apply(preprocess)

# Drop rows where preprocessing left an empty string
headlines = headlines[headlines['clean_text'].str.len() > 0].reset_index(drop=True)
print('Rows after preprocessing filter:', len(headlines))

## Section 4 — Temporal Train/Test Split

Strict cut at **2022-01-01**. The test period is the 2022 bear market plus the 2023 recovery — a regime change that genuinely tests out-of-sample generalisation.

In [ ]:
SPLIT_DATE = pd.Timestamp('2022-01-01')

train_h = headlines[headlines['date'] < SPLIT_DATE].reset_index(drop=True)
test_h = headlines[headlines['date'] >= SPLIT_DATE].reset_index(drop=True)

print(f'Train: {len(train_h):,} headlines from '
      f'{train_h["date"].min().date()} to {train_h["date"].max().date()}')
print(f'Test : {len(test_h):,} headlines from '
      f'{test_h["date"].min().date()} to {test_h["date"].max().date()}')
print('\nSentiment distribution (train):')
print(train_h['sentiment'].value_counts(normalize=True).round(3))
print('\nSentiment distribution (test):')
print(test_h['sentiment'].value_counts(normalize=True).round(3))

## Section 5 — TF-IDF Vectorisation and PCA Dimension Selection

TF-IDF with unigrams + bigrams, capped at 5000 features. We then reduce dimension via TruncatedSVD (the standard sparse-matrix equivalent of PCA — equivalent up to centring, which is negligible for high-dimensional sparse text) and pick the best `k` by 3-fold cross-validated accuracy of a Logistic Regression classifier.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)
X_train_tfidf = tfidf.fit_transform(train_h['clean_text'])
X_test_tfidf = tfidf.transform(test_h['clean_text'])

y_train = train_h['sentiment'].values
y_test = test_h['sentiment'].values

print('TF-IDF train shape:', X_train_tfidf.shape)
print('TF-IDF test  shape:', X_test_tfidf.shape)

In [ ]:
ks = [50, 100, 150, 200, 250, 300, 350, 400]
cv_scores = []

for k in ks:
    svd = TruncatedSVD(n_components=k, random_state=SEED)
    X_red = svd.fit_transform(X_train_tfidf)
    lr = LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1)
    scores = cross_val_score(lr, X_red, y_train, cv=3,
                             scoring='accuracy', n_jobs=-1)
    cv_scores.append(scores.mean())
    print(f'k={k:>3}  CV accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})')

best_k = ks[int(np.argmax(cv_scores))]
print(f'\nBest k = {best_k}')

In [ ]:
# Plot CV accuracy vs k
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ks, cv_scores, marker='o', linewidth=2)
ax.axvline(best_k, color='red', linestyle='--', alpha=0.5,
           label=f'Best k = {best_k}')
ax.set_xlabel('Number of components (k)')
ax.set_ylabel('3-fold CV accuracy')
ax.set_title('PCA dimension selection — cross-validated LogReg accuracy')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Final dimensionality reduction at best_k
svd = TruncatedSVD(n_components=best_k, random_state=SEED)
X_train_pca = svd.fit_transform(X_train_tfidf)
X_test_pca = svd.transform(X_test_tfidf)

print('Reduced train shape:', X_train_pca.shape)
print('Reduced test  shape:', X_test_pca.shape)
print(f'Explained variance ratio (sum): '
      f'{svd.explained_variance_ratio_.sum():.4f}')

## Section 6 — Benchmark Model: Logistic Regression

LogReg on the reduced features. This is the baseline classifier — its predictions become the input to all downstream signals.

In [ ]:
logreg = LogisticRegression(max_iter=2000, random_state=SEED, n_jobs=-1)
logreg.fit(X_train_pca, y_train)

train_pred_lr = logreg.predict(X_train_pca)
test_pred_lr = logreg.predict(X_test_pca)

acc_train_lr = accuracy_score(y_train, train_pred_lr)
acc_test_lr = accuracy_score(y_test, test_pred_lr)

print(f'LogReg train accuracy: {acc_train_lr:.4f}')
print(f'LogReg test  accuracy: {acc_test_lr:.4f}')
print('\n--- Classification report (test) ---')
print(classification_report(y_test, test_pred_lr))

In [ ]:
# Confusion matrix on test
labels_order = ['negative', 'neutral', 'positive']
cm_lr = confusion_matrix(y_test, test_pred_lr, labels=labels_order)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_lr, cmap='Blues')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(labels_order); ax.set_yticklabels(labels_order)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('LogReg confusion matrix (test)')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm_lr[i, j]), ha='center', va='center',
                color='white' if cm_lr[i, j] > cm_lr.max() / 2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

## Section 7 — Advanced Model: Multinomial Naive Bayes

NB is the natural alternative for short-text classification — it works on raw count features rather than TF-IDF/PCA.

> **Note on model choice**: Once we move to daily aggregation, classifier accuracy gains of a few percentage points have limited impact on downstream strategy performance. What matters is signal construction, not classifier accuracy. We therefore use **LogReg predictions** for all subsequent signals; NB is reported here as a sanity check.

In [ ]:
# Use raw counts for NB
cv_vec = CountVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)
X_train_cv = cv_vec.fit_transform(train_h['clean_text'])
X_test_cv = cv_vec.transform(test_h['clean_text'])

nb = MultinomialNB()
nb.fit(X_train_cv, y_train)

train_pred_nb = nb.predict(X_train_cv)
test_pred_nb = nb.predict(X_test_cv)

acc_train_nb = accuracy_score(y_train, train_pred_nb)
acc_test_nb = accuracy_score(y_test, test_pred_nb)

print(f'NB train accuracy: {acc_train_nb:.4f}')
print(f'NB test  accuracy: {acc_test_nb:.4f}')
print('\n--- Classification report (test) ---')
print(classification_report(y_test, test_pred_nb))

## Section 8 — NLP Visualisations

Seven required visualisations covering classifier performance, feature interpretation, structure, and clustering.

### 8.1 Accuracy bar chart (LogReg vs NB, train vs test)

In [ ]:
acc_data = pd.DataFrame({
    'Model': ['LogReg', 'LogReg', 'Naive Bayes', 'Naive Bayes'],
    'Split': ['Train', 'Test', 'Train', 'Test'],
    'Accuracy': [acc_train_lr, acc_test_lr, acc_train_nb, acc_test_nb],
})
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2)
width = 0.35
train_vals = acc_data[acc_data['Split'] == 'Train']['Accuracy'].values
test_vals = acc_data[acc_data['Split'] == 'Test']['Accuracy'].values
ax.bar(x - width/2, train_vals, width, label='Train', color='steelblue')
ax.bar(x + width/2, test_vals, width, label='Test', color='coral')
ax.set_xticks(x); ax.set_xticklabels(['LogReg', 'Naive Bayes'])
ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1)
ax.set_title('Classifier accuracy — train vs test')
for xi, (tr, te) in enumerate(zip(train_vals, test_vals)):
    ax.text(xi - width/2, tr + 0.01, f'{tr:.3f}', ha='center')
    ax.text(xi + width/2, te + 0.01, f'{te:.3f}', ha='center')
ax.legend(); plt.tight_layout(); plt.show()

### 8.2 Top TF-IDF terms per sentiment class

For each class, show the top 15 features by LogReg coefficient (most diagnostic of that class).

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())

# Project the LogReg coefficients (in PCA space) back to TF-IDF feature space
# coef shape: (n_classes, n_components); svd.components_ shape: (n_components, n_features)
coef_features = logreg.coef_ @ svd.components_  # (n_classes, n_features)

class_labels = list(logreg.classes_)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, lbl in enumerate(class_labels):
    top_idx = np.argsort(coef_features[i])[-15:]
    top_terms = feature_names[top_idx]
    top_vals = coef_features[i][top_idx]
    axes[i].barh(range(len(top_terms)), top_vals, color='steelblue')
    axes[i].set_yticks(range(len(top_terms)))
    axes[i].set_yticklabels(top_terms)
    axes[i].set_title(f'Top terms — {lbl}')
    axes[i].set_xlabel('LogReg coefficient (projected)')
plt.tight_layout(); plt.show()

### 8.3 PCA 2D scatter coloured by predicted sentiment

A random sample of 5,000 test headlines projected onto the first two components.

In [ ]:
rng = np.random.default_rng(SEED)
n_plot = 5000
idx = rng.choice(X_test_pca.shape[0], size=n_plot, replace=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = {'negative': 'red', 'neutral': 'gray', 'positive': 'green'}
for lbl in ['neutral', 'positive', 'negative']:
    mask = test_pred_lr[idx] == lbl
    ax.scatter(X_test_pca[idx][mask, 0], X_test_pca[idx][mask, 1],
               c=colors[lbl], label=lbl, alpha=0.4, s=10)
ax.set_xlabel('Component 1'); ax.set_ylabel('Component 2')
ax.set_title('Test headlines — first 2 PCA components, coloured by LogReg prediction')
ax.legend(); plt.tight_layout(); plt.show()

### 8.4 K-Means clustering on TF-IDF features (k=10)

Cluster training headlines into 10 topical groups. Show the size of each cluster and 3 representative headlines (closest to centroid).

In [ ]:
K = 10
kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10)
cluster_train = kmeans.fit_predict(X_train_pca)
cluster_test = kmeans.predict(X_test_pca)

train_h = train_h.copy()
test_h = test_h.copy()
train_h['cluster'] = cluster_train
test_h['cluster'] = cluster_test

cluster_sizes = pd.Series(cluster_train).value_counts().sort_index()
print('Cluster sizes (training):')
print(cluster_sizes.to_string())

# Find 3 representative headlines per cluster (closest to centroid)
print('\n--- Representative headlines per cluster ---')
for c in range(K):
    mask = cluster_train == c
    if mask.sum() == 0:
        continue
    centroid = kmeans.cluster_centers_[c]
    dist = np.linalg.norm(X_train_pca[mask] - centroid, axis=1)
    nearest_local = np.argsort(dist)[:3]
    nearest_global = np.where(mask)[0][nearest_local]
    print(f'\nCluster {c} (n={mask.sum()})')
    for gi in nearest_global:
        print(f'   - {train_h.iloc[gi]["headline"]}')

### 8.5 Sentiment distribution per cluster (stacked bar)

In [ ]:
cluster_sent = train_h.groupby('cluster')['sentiment'].value_counts(
    normalize=True
).unstack(fill_value=0)
cluster_sent = cluster_sent[['negative', 'neutral', 'positive']]

fig, ax = plt.subplots(figsize=(11, 5))
cluster_sent.plot(kind='bar', stacked=True, ax=ax,
                  color=['red', 'gray', 'green'])
ax.set_xlabel('Cluster'); ax.set_ylabel('Share')
ax.set_title('Sentiment distribution per K-Means cluster (training)')
ax.legend(title='Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

### 8.6 Classification metrics table

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def metrics_table(y_true, y_pred, model_name, split):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    return {'Model': model_name, 'Split': split,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': p, 'Recall': r, 'F1': f}

metrics_rows = [
    metrics_table(y_train, train_pred_lr, 'LogReg', 'Train'),
    metrics_table(y_test,  test_pred_lr,  'LogReg', 'Test'),
    metrics_table(y_train, train_pred_nb, 'NaiveBayes', 'Train'),
    metrics_table(y_test,  test_pred_nb,  'NaiveBayes', 'Test'),
]
metrics_df = pd.DataFrame(metrics_rows).round(4)
print(metrics_df.to_string(index=False))

### 8.7 Most confident correct and incorrect predictions

Use `predict_proba` from LogReg. For each class, show the highest-confidence correct prediction and the highest-confidence wrong prediction on the test set.

In [ ]:
probs_test = logreg.predict_proba(X_test_pca)
max_p = probs_test.max(axis=1)
correct = (test_pred_lr == y_test)

print('=== Most confident CORRECT (top per class) ===')
for cls in logreg.classes_:
    mask = correct & (test_pred_lr == cls)
    if mask.sum() == 0:
        continue
    best_local = np.argmax(np.where(mask, max_p, -1))
    print(f'\n[{cls}] confidence={max_p[best_local]:.3f}')
    print(f'  {test_h.iloc[best_local]["headline"]}')

print('\n=== Most confident INCORRECT (top per class) ===')
wrong = ~correct
for cls in logreg.classes_:
    mask = wrong & (test_pred_lr == cls)
    if mask.sum() == 0:
        continue
    best_local = np.argmax(np.where(mask, max_p, -1))
    print(f'\n[predicted {cls}, actual {y_test[best_local]}] '
          f'confidence={max_p[best_local]:.3f}')
    print(f'  {test_h.iloc[best_local]["headline"]}')

## Section 9 — Classify All Headlines

We re-classify every headline (train + test) with the trained LogReg, so daily signals can be aggregated over the full sample. We also keep the NB predictions for an agreement check.

> **Circularity caveat**: LogReg was trained on the full training set, so training-period strategy metrics constructed from these labels are upper bounds. Test-period metrics are not affected.

In [ ]:
# Stack train + test (already classified above), keep date order
train_h['pred_lr'] = train_pred_lr
train_h['pred_nb'] = train_pred_nb
test_h['pred_lr']  = test_pred_lr
test_h['pred_nb']  = test_pred_nb
all_h = pd.concat([train_h, test_h], ignore_index=True)

agreement = (all_h['pred_lr'] == all_h['pred_nb']).mean()
print(f'LogReg vs NB agreement rate: {agreement:.4f}')

agreement_by_cls = (
    all_h.groupby('pred_lr')
        .apply(lambda d: (d['pred_lr'] == d['pred_nb']).mean())
        .round(3)
)
print('\nAgreement rate by LogReg class:')
print(agreement_by_cls.to_string())

## Section 10 — Daily Signals and Market Merge

Construct exactly two daily aggregate signals from the LogReg predictions:

- **S_t** = (N_pos − N_neg) / N_total  — net sentiment (the mandatory baseline)
- **S_neg_t** = N_neg / N_total       — negative fraction (for the asymmetry hypothesis)

Then merge with SPX returns and create forward-return columns. The convention throughout: signal observed at end of day *t* trades the open-to-open or close-to-close return of day *t+1*. We use `next_day_return = sprtrn.shift(-1)`.

In [ ]:
# Daily counts of predicted classes
daily_counts = (
    all_h.groupby([pd.Grouper(key='date', freq='D'), 'pred_lr'])
        .size().unstack(fill_value=0)
)
for col in ['negative', 'neutral', 'positive']:
    if col not in daily_counts.columns:
        daily_counts[col] = 0
daily_counts['N_total'] = daily_counts.sum(axis=1)

# Construct the two signals
daily = pd.DataFrame(index=daily_counts.index)
daily['N_pos'] = daily_counts['positive']
daily['N_neg'] = daily_counts['negative']
daily['N_neu'] = daily_counts['neutral']
daily['N_total'] = daily_counts['N_total']
daily['S_t'] = (daily['N_pos'] - daily['N_neg']) / daily['N_total'].replace(0, np.nan)
daily['S_neg_t'] = daily['N_neg'] / daily['N_total'].replace(0, np.nan)
daily = daily.reset_index()

print('Daily signal frame head:')
print(daily.head())
print('\nN_total summary:')
print(daily['N_total'].describe().round(1))

In [ ]:
# Merge with SPX (inner join on trading days only)
mkt = daily.merge(spx, on='date', how='inner')

# Forward returns
mkt['next_day_return'] = mkt['sprtrn'].shift(-1)
mkt['abs_return'] = mkt['sprtrn'].abs()
mkt['fwd_5d'] = mkt['sprtrn'].shift(-1).rolling(5).sum().shift(-4)
mkt['fwd_10d'] = mkt['sprtrn'].shift(-1).rolling(10).sum().shift(-9)
mkt['fwd_20d'] = mkt['sprtrn'].shift(-1).rolling(20).sum().shift(-19)

# Forward-fill any single-day signal gaps (rare weekend-news cases)
for c in ['S_t', 'S_neg_t']:
    mkt[c] = mkt[c].ffill()

print('Merged market frame:')
print(mkt[['date', 'S_t', 'S_neg_t', 'sprtrn', 'next_day_return']].head())
print(f'\nRows: {len(mkt)}, date range: '
      f'{mkt["date"].min().date()} to {mkt["date"].max().date()}')

In [ ]:
# Persist train/test masks
train_mask = mkt['date'] < SPLIT_DATE
test_mask = mkt['date'] >= SPLIT_DATE
print(f'Train days: {train_mask.sum()}   Test days: {test_mask.sum()}')

## Section 10A — Statistical Tests of Predictive Power

**Before** building any strategies we ask: do the signals carry predictive information at all? Five tests:

1. Spearman IC vs next-day return (train and test separately)
2. Quintile analysis: sort training days by S_neg_t, report mean annualised return per quintile
3. Volatility prediction: Spearman IC of S_neg_t vs |return|
4. Signal decay: IC at horizons 1d, 5d, 10d, 20d
5. Rolling 60-day IC plot

### 10A.1 Spearman IC vs next-day return

In [ ]:
def spearman_ic(s, r):
    s = pd.Series(s); r = pd.Series(r)
    valid = s.notna() & r.notna()
    if valid.sum() < 30:
        return np.nan, np.nan
    rho, p = stats.spearmanr(s[valid], r[valid])
    return rho, p

rows = []
for sig in ['S_t', 'S_neg_t']:
    for split, m in [('Train', train_mask), ('Test', test_mask)]:
        rho, p = spearman_ic(mkt.loc[m, sig], mkt.loc[m, 'next_day_return'])
        rows.append({'Signal': sig, 'Split': split,
                     'Spearman IC': round(rho, 4), 'p-value': round(p, 4)})
ic_df = pd.DataFrame(rows)
print(ic_df.to_string(index=False))

### 10A.2 Quintile analysis (training period)

In [ ]:
train_df = mkt.loc[train_mask].copy()
train_df['S_neg_quintile'] = pd.qcut(
    train_df['S_neg_t'], 5, labels=['Q1 low', 'Q2', 'Q3', 'Q4', 'Q5 high']
)
quint = train_df.groupby('S_neg_quintile')['next_day_return'].agg(
    ['mean', 'std', 'count']
)
quint['annualised_return'] = quint['mean'] * 252
quint['annualised_vol'] = quint['std'] * np.sqrt(252)
print(quint.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
quint['annualised_return'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_ylabel('Annualised next-day return')
ax.set_xlabel('S_neg_t quintile')
ax.set_title('Quintile analysis — training period')
ax.axhline(0, color='black', lw=0.8)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

test_df = mkt.loc[test_mask].copy()
test_df['S_neg_quintile'] = pd.qcut(
    test_df['S_neg_t'], 5, labels=['Q1 low', 'Q2', 'Q3', 'Q4', 'Q5 high']
)
quint_test = test_df.groupby('S_neg_quintile')['next_day_return'].agg(
    ['mean', 'std', 'count']
)
quint_test['annualised_return'] = quint_test['mean'] * 252
quint_test['annualised_vol'] = quint_test['std'] * np.sqrt(252)
print('\n=== Test period quintiles ===')
print(quint_test.round(4))

q5_train_sign = '+' if quint['mean'].iloc[-1] > 0 else '-'
q5_test_sign  = '+' if quint_test['mean'].iloc[-1] > 0 else '-'
print(f'\nQ5 (highest negativity) sign — train: {q5_train_sign}  test: {q5_test_sign}  '
      f'consistent: {q5_train_sign == q5_test_sign}')


Q5 is the highest-negativity quintile. If both periods show the same sign on Q5 mean, the high-negativity → directional return relationship is consistent out-of-sample — relevant to the asymmetry hypothesis. (Read the printed `consistent:` flag above.)


### 10A.3 Volatility prediction (S_neg_t vs |return|)

If sentiment carries any market-relevant information, we expect it to track
*uncertainty* even if it doesn't time direction. We test this with the
Spearman IC of S_neg_t against next-day absolute return.

> **Read this carefully alongside the printed table.** The training-period
> result is strong: S_neg_t IC vs |r_{t+1}| = **+0.13, p<0.001**. The
> test-period result is much weaker: **+0.045, p≈0.31** — directionally
> consistent but not statistically distinguishable from zero on the
> 24-month test sample. The "sentiment predicts volatility" claim is
> therefore *robust on training, fragile out-of-sample*. We treat this as
> the same regime-instability story that the directional quintile flip
> (Section 10A.2) tells: the sentiment–market linkage weakens or inverts
> in the 2022–2023 sample. The negative_risk_off strategy still works on
> test (significant positive alpha), but it works because the *directional*
> sign held — not because the volatility-tracking finding generalised.

In [ ]:
rows = []
for sig in ['S_t', 'S_neg_t']:
    for split, m in [('Train', train_mask), ('Test', test_mask)]:
        rho, p = spearman_ic(mkt.loc[m, sig], mkt.loc[m, 'abs_return'].shift(-1))
        rows.append({'Signal': sig, 'Split': split,
                     'IC vs |r_{t+1}|': round(rho, 4), 'p-value': round(p, 4)})
print(pd.DataFrame(rows).to_string(index=False))

### 10A.4 Signal decay — IC at multiple horizons

In [ ]:
horizons = [('1d', 'next_day_return'),
            ('5d', 'fwd_5d'),
            ('10d', 'fwd_10d'),
            ('20d', 'fwd_20d')]
rows = []
for sig in ['S_t', 'S_neg_t']:
    for split, m in [('Train', train_mask), ('Test', test_mask)]:
        for hname, hcol in horizons:
            rho, p = spearman_ic(mkt.loc[m, sig], mkt.loc[m, hcol])
            rows.append({'Signal': sig, 'Split': split, 'Horizon': hname,
                         'IC': round(rho, 4), 'p-value': round(p, 4)})
decay_df = pd.DataFrame(rows)
print(decay_df.to_string(index=False))

### 10A.5 Rolling 60-day IC

In [ ]:
window = 60
roll_ic = pd.DataFrame({'date': mkt['date']})
for sig in ['S_t', 'S_neg_t']:
    ic_series = []
    for i in range(len(mkt)):
        if i < window:
            ic_series.append(np.nan); continue
        s_win = mkt[sig].iloc[i - window:i]
        r_win = mkt['next_day_return'].iloc[i - window:i]
        valid = s_win.notna() & r_win.notna()
        if valid.sum() < 30:
            ic_series.append(np.nan); continue
        rho, _ = stats.spearmanr(s_win[valid], r_win[valid])
        ic_series.append(rho)
    roll_ic[sig] = ic_series

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(roll_ic['date'], roll_ic['S_t'], label='S_t', alpha=0.8)
ax.plot(roll_ic['date'], roll_ic['S_neg_t'], label='S_neg_t', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.axvline(SPLIT_DATE, color='red', linestyle='--', alpha=0.6,
           label='Train/Test split')
ax.set_title(f'Rolling {window}-day Spearman IC vs next-day return')
ax.set_ylabel('Spearman rho')
ax.legend(); plt.tight_layout(); plt.show()

## Section 10B — Parameter Choices

We commit to two parameters BEFORE any strategy backtest. These are chosen by **financial convention** rather than optimised on data, which avoids data-snooping bias:

- **WINDOW = 20** trading days ≈ one calendar month — the standard look-back unit in financial time series.
- **THRESHOLD = 1.0** standard deviation — the canonical "one-sigma" event marker in statistics and risk management.

No grid search is performed. These choices are fixed for the entire backtest.

In [ ]:
WINDOW = 20
THRESHOLD = 1.0
print(f'WINDOW = {WINDOW} trading days (~1 month)')
print(f'THRESHOLD = {THRESHOLD} standard deviations')

## Section 10C — Diagnostic: why the naive rules from the brief are degenerate

The project brief's default starting point for Step 2 is:

  - **Momentum**: w_t = 1 if S_t > 0, else 0
  - **Mean-reversion**: w_t = 1 if S_t < 0, else 0

Both threshold S_t at zero. WSJ headline sentiment skews structurally
negative (financial news bias plus the well-documented negative skew of
FinBERT on financial text), so S_t has a persistently negative mean.
Consequence: momentum sits in cash almost every day; mean-reversion sits
long almost every day. Neither tests the actual hypothesis the rule is
named after.

The diagnostic below quantifies this. The headline numbers — momentum
invested ≈ 3% of days, mean-reversion ≈ 97% — motivate the redesigned
strategies in Section 11, where both rules threshold against the signal's
*own recent distribution* rather than an unconditional zero.

In [ ]:
# Diagnostic: naive momentum and mean-reversion from the project brief.
# These rules are NOT used downstream — they exist only to motivate the
# redesigned strategies in Section 11.

# Bring rf in locally so this cell runs regardless of where it sits in
# the notebook (the main rf merge happens in Section 12).
_mkt = mkt.merge(ff3[['date', 'rf']], on='date', how='left')
_mkt['rf'] = _mkt['rf'].fillna(0.0)

w_naive_mom    = (_mkt['S_t'] > 0).astype(float)
w_naive_revert = (_mkt['S_t'] < 0).astype(float)

print('S_t distribution (full sample):')
print(_mkt['S_t'].describe().round(4).to_string())

print('\nDays invested under naive rules:')
print(f'  naive momentum     (S_t > 0):  {w_naive_mom.mean()*100:5.1f}%')
print(f'  naive mean-reversion (S_t < 0): {w_naive_revert.mean()*100:5.1f}%')

print('\nNaive-rule performance (Sharpe, annualised):')
header = f'  {"strategy":<22} {"split":<6} {"days_inv":>9} {"ann_return":>11} {"sharpe":>8}'
print(header)
print('  ' + '-' * (len(header) - 2))

for name, w in [('naive_momentum', w_naive_mom),
                ('naive_meanrev',  w_naive_revert),
                ('buy_hold',       pd.Series(1.0, index=_mkt.index))]:
    w_shift = w.shift(1)
    port_ret = w_shift * _mkt['sprtrn'] + (1 - w_shift) * _mkt['rf']
    for split, m in [('Train', _mkt['date'] < SPLIT_DATE),
                     ('Test',  _mkt['date'] >= SPLIT_DATE)]:
        sub = port_ret[m.values].dropna()
        sub_rf = _mkt['rf'][m.values].loc[sub.index]
        days_inv = w_shift[m.values].mean() * 100
        cum = (1 + sub).prod() - 1
        ann = (1 + cum) ** (252 / len(sub)) - 1
        excess = sub - sub_rf
        ann_vol = sub.std() * np.sqrt(252)
        sharpe = (excess.mean() * 252) / ann_vol if ann_vol > 0 else np.nan
        print(f'  {name:<22} {split:<6} {days_inv:>8.1f}% {ann:>10.2%} {sharpe:>8.3f}')

## Section 11 – Strategies

Four sentiment-based rules plus two benchmarks. All weights are in [0, 1] and all signals on day *t* are observed BEFORE trading day *t+1* – no lookahead.

| Strategy | Signal | Rule |
|----------|--------|------|
| trend_momentum | S_t | go long when S_t exceeds its trailing 20-day EMA — ride improving sentiment |
| negativity_contrarian | S_neg_t | go long when negativity z-score >1σ above its trailing 20-day mean — buy the oversold bounce |
| surprise | S_t | go long when today's sentiment exceeds its trailing 20-day mean — level above baseline |
| negative_risk_off | S_neg_t | move to cash on the same negativity spike — opposite action to contrarian, the horse race that tests the asymmetry |


In [ ]:
def rule_trend_momentum(S):
    ema = S.shift(1).ewm(span=WINDOW, adjust=False).mean()
    delta = S - ema
    return (delta > 0).astype(float)

def rule_negativity_contrarian(S_neg):
    prior = S_neg.shift(1)
    mu  = prior.rolling(WINDOW, min_periods=WINDOW // 2).mean()
    sig = prior.rolling(WINDOW, min_periods=WINDOW // 2).std().clip(lower=1e-4)
    z = (S_neg - mu) / sig
    return (z > THRESHOLD).astype(float)

def rule_surprise(S):
    baseline = S.shift(1).rolling(WINDOW, min_periods=WINDOW // 2).mean()
    return (S > baseline).astype(float)

def rule_negative_risk_off(S):
    prior = S.shift(1)
    mu = prior.rolling(WINDOW, min_periods=WINDOW // 2).mean()
    sigma = prior.rolling(WINDOW, min_periods=WINDOW // 2).std().clip(lower=1e-4)
    spike = S > (mu + THRESHOLD * sigma)
    return pd.Series(np.where(spike, 0.0, 1.0), index=S.index)

strategy_configs = [
    ('trend_momentum',          'S_t',     rule_trend_momentum,        'w_trend'),
    ('negativity_contrarian',   'S_neg_t', rule_negativity_contrarian, 'w_contra'),
    ('surprise',                'S_t',     rule_surprise,              'w_surprise'),
    ('negative_risk_off',       'S_neg_t', rule_negative_risk_off,     'w_neg_off'),
]

# Build weight columns
for name, sig_col, rule, w_col in strategy_configs:
    mkt[w_col] = rule(mkt[sig_col])

# Benchmarks
mkt['w_buyhold'] = 1.0
mkt['w_5050'] = 0.5

weight_cols = [w for *_, w in strategy_configs] + ['w_buyhold', 'w_5050']
strategy_names = [n for n, *_ in strategy_configs] + ['buy_hold', 'fifty_fifty']

# Diagnostic: percent days invested
print('Percent of days with weight > 0 (full sample):')
for name, w in zip(strategy_names, weight_cols):
    pct = (mkt[w] > 0).mean() * 100
    print(f'  {name:<22s}  {pct:5.1f}%')

## Section 12 — Backtest Engine

Portfolio return:

  r_pf,t+1 = w_t · r_SPX,t+1 + (1 − w_t) · r_f,t+1

Metrics computed on each strategy × period:
- cumulative return
- annualised return (compounded)
- annualised Sharpe (excess of risk-free, ×√252)
- maximum drawdown
- Calmar ratio (annualised return / |max drawdown|)

In [ ]:
# Bring r_f in from FF3
mkt = mkt.merge(ff3[['date', 'rf']], on='date', how='left')
mkt['rf'] = mkt['rf'].fillna(0.0)

def compute_portfolio_returns(weights, mkt_returns, rf):
    return weights * mkt_returns + (1 - weights) * rf


def perf_metrics(returns, rf):
    returns = returns.dropna()
    if len(returns) < 2:
        return {'cum_return': np.nan, 'ann_return': np.nan,
                'sharpe': np.nan, 'max_dd': np.nan, 'calmar': np.nan}
    cum = (1 + returns).prod() - 1
    n_days = len(returns)
    ann_ret = (1 + cum) ** (252 / n_days) - 1
    excess = returns - rf.loc[returns.index]
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (excess.mean() * 252) / ann_vol if ann_vol > 0 else np.nan
    equity = (1 + returns).cumprod()
    rolling_max = equity.cummax()
    dd = (equity / rolling_max - 1).min()
    calmar = ann_ret / abs(dd) if dd < 0 else np.nan
    return {'cum_return': cum, 'ann_return': ann_ret,
            'sharpe': sharpe, 'max_dd': dd, 'calmar': calmar}


def run_backtest(mkt, weight_cols, strategy_names):
    rows = []
    for name, w_col in zip(strategy_names, weight_cols):
        # Shift weight by 1 so that weight at t-1 trades return at t
        # We've defined w on day t and earn return on day t+1; equivalently
        # shift weight forward by one and use sprtrn on the same day.
        w_shift = mkt[w_col].shift(1)
        port_ret = compute_portfolio_returns(w_shift, mkt['sprtrn'], mkt['rf'])
        port_ret.index = mkt['date']
        rf_idx = mkt.set_index('date')['rf']
        for split, m in [('Train', mkt['date'] < SPLIT_DATE),
                         ('Test',  mkt['date'] >= SPLIT_DATE)]:
            sub_ret = port_ret[m.values]
            sub_rf = rf_idx[m.values]
            mets = perf_metrics(sub_ret, sub_rf)
            rows.append({'strategy': name, 'split': split, **mets,
                         'returns': sub_ret})
    return pd.DataFrame(rows)


results_df = run_backtest(mkt, weight_cols, strategy_names)

# Display table without the bulky returns column
display_cols = ['strategy', 'split', 'cum_return', 'ann_return',
                'sharpe', 'max_dd', 'calmar']
print(results_df[display_cols].round(4).to_string(index=False))

## Section 13 — Required Visualisations

The three plots required by the project specification:

1. **Sentiment time series overlaid with SPX returns** — 20-day rolling
   mean of S_t and S_neg_t in the top panel, SPX cumulative return in the
   bottom panel, with the train/test split marked.
2. **Scatter: sentiment vs next-day return (with R²)** — slope, R², and
   Pearson r reported in each subtitle. R² is small in absolute terms
   (~10⁻³), consistent with the small Spearman ICs in Section 10A.1 and
   the well-established result that aggregate sentiment has weak daily-
   horizon directional power.
3. **Cumulative return: strategy vs buy-and-hold vs 50/50** — log-scale
   equity curves for all four sentiment strategies plus both benchmarks.

Drawdown curves are also included as a supporting (non-required) plot.

In [ ]:
# 1. Cumulative return
fig, ax = plt.subplots(figsize=(12, 6))
for name, w_col in zip(strategy_names, weight_cols):
    w_shift = mkt[w_col].shift(1)
    port_ret = (w_shift * mkt['sprtrn'] + (1 - w_shift) * mkt['rf']).fillna(0)
    equity = (1 + port_ret).cumprod()
    ax.plot(mkt['date'], equity, label=name, linewidth=1.4)
ax.axvline(SPLIT_DATE, color='red', linestyle='--', alpha=0.6,
           label='Train/Test split')
ax.set_yscale('log')
ax.set_ylabel('Equity (log scale)'); ax.set_xlabel('Date')
ax.set_title('Cumulative return — all strategies')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 2. Drawdown
fig, ax = plt.subplots(figsize=(12, 5))
for name, w_col in zip(strategy_names, weight_cols):
    w_shift = mkt[w_col].shift(1)
    port_ret = (w_shift * mkt['sprtrn'] + (1 - w_shift) * mkt['rf']).fillna(0)
    equity = (1 + port_ret).cumprod()
    dd = equity / equity.cummax() - 1
    ax.plot(mkt['date'], dd, label=name, linewidth=1.0, alpha=0.85)
ax.axvline(SPLIT_DATE, color='red', linestyle='--', alpha=0.6)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Drawdown'); ax.set_xlabel('Date')
ax.set_title('Drawdown — all strategies')
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Required Plot #1: Sentiment time series overlaid with SPX cumulative return
# Two stacked panels share the x-axis: smoothed sentiment on top, SPX equity
# on the bottom, with the train/test split marked. Smoothing uses a 20-day
# rolling mean to make regimes visible (raw daily sentiment is too noisy
# for visual interpretation).

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Top panel — sentiment (smoothed)
s_t_smooth = mkt['S_t'].rolling(20, min_periods=10).mean()
s_neg_smooth = mkt['S_neg_t'].rolling(20, min_periods=10).mean()
ax_top.plot(mkt['date'], s_t_smooth, label='S_t (20d MA)',
            color='steelblue', linewidth=1.4)
ax_top_r = ax_top.twinx()
ax_top_r.plot(mkt['date'], s_neg_smooth, label='S_neg_t (20d MA)',
              color='darkorange', linewidth=1.4, alpha=0.85)
ax_top.axvline(SPLIT_DATE, color='red', linestyle='--', alpha=0.6)
ax_top.axhline(0, color='black', lw=0.5)
ax_top.set_ylabel('S_t', color='steelblue')
ax_top_r.set_ylabel('S_neg_t', color='darkorange')
ax_top.set_title('Aggregate WSJ sentiment (20-day rolling) vs SPX')
# Combined legend
lines1, labels1 = ax_top.get_legend_handles_labels()
lines2, labels2 = ax_top_r.get_legend_handles_labels()
ax_top.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

# Bottom panel — SPX equity curve
spx_equity = (1 + mkt['sprtrn'].fillna(0)).cumprod()
ax_bot.plot(mkt['date'], spx_equity, color='black', linewidth=1.4,
            label='SPX cumulative return')
ax_bot.axvline(SPLIT_DATE, color='red', linestyle='--', alpha=0.6,
               label='Train/Test split')
ax_bot.set_ylabel('SPX equity (start = 1.0)')
ax_bot.set_xlabel('Date')
ax_bot.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Required Plot #2 (per PDF): Scatter sentiment vs next-day return with R²
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, sig in zip(axes, ['S_t', 'S_neg_t']):
    valid = mkt[[sig, 'next_day_return']].dropna()
    x = valid[sig].values
    y = valid['next_day_return'].values

    # Linear fit
    slope, intercept = np.polyfit(x, y, 1)
    y_hat = slope * x + intercept
    ss_res = ((y - y_hat) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    r_squared = 1 - ss_res / ss_tot

    # Pearson r and its p-value
    pearson_r, pearson_p = stats.pearsonr(x, y)

    ax.scatter(x, y, alpha=0.3, s=8, color='steelblue')
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, slope * xs + intercept, color='red', lw=1.5)
    ax.axhline(0, color='black', lw=0.5)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_xlabel(sig)
    ax.set_ylabel('Next-day return')
    ax.set_title(
        f'{sig} vs next-day return\n'
        f'slope={slope:.4f}  R²={r_squared:.4f}  '
        f'Pearson r={pearson_r:.3f} (p={pearson_p:.3f})'
    )
plt.tight_layout()
plt.show()

## Section 14 — Fama-French 3-Factor Alpha

Regress strategy excess returns on the three factors with **HAC (Newey-West) standard errors**, maxlags=5:

  r_excess = α + β1·MKT + β2·SMB + β3·HML + ε

We report annualised α, t-stat, and p-value. The constant `α > 0` with `t > 1.96` is the textbook test of "skill beyond factor exposure".

> **Footnote on the buy-and-hold alpha.** Buy-and-hold posts a small but
> statistically significant *negative* FF3 alpha (≈ −1.6% annualised,
> p≈0.05 on test; ≈ −1.8%, p<0.001 on train). This is a benchmark
> composition artefact, not a finding about the SPX. The `mktrf` factor is
> the **CRSP value-weighted market excess return** — a broader basket
> including small-caps and microcaps — while our backtest uses the S&P 500
> total return (`sprtrn`). SPX has β≈0.99 to MKT but slightly underperforms
> the broader CRSP universe over this sample, and that small persistent
> drag shows up as negative alpha after SMB/HML absorb the size and value
> tilts. Sentiment-strategy alphas should be read **relative to the
> buy-and-hold alpha** as a reference point, not against zero.

In [ ]:
def ff3_alpha(strategy_returns, ff3_df, split_mask, dates):
    df = pd.DataFrame({
        'r': strategy_returns.values,
        'date': dates.values,
    }).dropna()
    df = df.merge(ff3_df, on='date', how='inner')
    df = df[df['date'].isin(dates[split_mask])]
    if len(df) < 30:
        return {'alpha_ann': np.nan, 'tstat': np.nan, 'pvalue': np.nan,
                'beta_mkt': np.nan, 'beta_smb': np.nan, 'beta_hml': np.nan}
    df['excess'] = df['r'] - df['rf']
    X = sm.add_constant(df[['mktrf', 'smb', 'hml']])
    y = df['excess']
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    return {
        'alpha_ann': model.params['const'] * 252,
        'tstat': model.tvalues['const'],
        'pvalue': model.pvalues['const'],
        'beta_mkt': model.params['mktrf'],
        'beta_smb': model.params['smb'],
        'beta_hml': model.params['hml'],
    }


ff3_rows = []
for name, w_col in zip(strategy_names, weight_cols):
    w_shift = mkt[w_col].shift(1)
    port_ret = w_shift * mkt['sprtrn'] + (1 - w_shift) * mkt['rf']
    for split, m in [('Train', mkt['date'] < SPLIT_DATE),
                     ('Test',  mkt['date'] >= SPLIT_DATE)]:
        res = ff3_alpha(port_ret, ff3, m.values, mkt['date'])
        ff3_rows.append({'strategy': name, 'split': split, **res})

ff3_df_out = pd.DataFrame(ff3_rows)
print(ff3_df_out[['strategy', 'split', 'alpha_ann', 'tstat',
                  'pvalue', 'beta_mkt']].round(4).to_string(index=False))

## Section 15 — Topic Analysis

Daily share of each K-Means cluster as a regressor for next-day return. This asks: do certain topics carry directional information beyond aggregate sentiment?

In [ ]:
# Daily cluster proportions
all_h_clu = pd.concat([train_h, test_h], ignore_index=True)
cluster_daily = (
    all_h_clu.groupby([pd.Grouper(key='date', freq='D'), 'cluster'])
        .size().unstack(fill_value=0)
)
cluster_daily = cluster_daily.div(cluster_daily.sum(axis=1), axis=0)
cluster_daily.columns = [f'clu_{c}' for c in cluster_daily.columns]
cluster_daily = cluster_daily.reset_index()

mkt_clu = mkt.merge(cluster_daily, on='date', how='left')

# Filter thresholds — clusters too small or too low-variance produce
# unstable t-stats (cluster 9 with n=239 was returning t≈32 from leverage
# on 1-2 nonzero days)
MIN_TOTAL_HEADLINES = 500
MIN_DAILY_SHARE_STD = 0.005

cluster_total_n = (all_h_clu['cluster'].value_counts()
                   .reindex(range(K), fill_value=0))

topic_rows = []
for c in range(K):
    col = f'clu_{c}'
    if cluster_total_n[c] < MIN_TOTAL_HEADLINES:
        topic_rows.append({
            'cluster': c, 'split': '—',
            'coef': np.nan, 'tstat': np.nan, 'pvalue': np.nan,
            'note': f'skipped: n={cluster_total_n[c]} < {MIN_TOTAL_HEADLINES}',
        })
        continue
    for split, m in [('Train', mkt_clu['date'] < SPLIT_DATE),
                     ('Test',  mkt_clu['date'] >= SPLIT_DATE)]:
        sub = mkt_clu.loc[m, [col, 'S_t', 'next_day_return']].dropna()
        if len(sub) < 30 or sub[col].std() < MIN_DAILY_SHARE_STD:
            topic_rows.append({
                'cluster': c, 'split': split,
                'coef': np.nan, 'tstat': np.nan, 'pvalue': np.nan,
                'note': f'skipped: low variance (std={sub[col].std():.4f})',
            })
            continue
        X = sm.add_constant(sub[[col, 'S_t']])
        model = sm.OLS(sub['next_day_return'], X).fit(
            cov_type='HAC', cov_kwds={'maxlags': 5}
        )
        topic_rows.append({
            'cluster': c, 'split': split,
            'coef': model.params[col],
            'tstat': model.tvalues[col],
            'pvalue': model.pvalues[col],
            'note': '',
        })
topic_df = pd.DataFrame(topic_rows)
print(topic_df.round(4).to_string(index=False))


## Section 16 — Error Analysis

The 10 worst days for the **negativity_contrarian** strategy on the test
set (largest negative portfolio returns), with the headlines from those
days. Contrarian is the most informative failure case in the four-strategy
suite: it took *long* positions on negativity spikes — the opposite of
risk_off — and 2022 punished that bet. These worst days show why the
"buy-the-fear" interpretation of the asymmetry hypothesis fails out-of-
sample.


In [ ]:
w_contra_shift = mkt['w_contra'].shift(1)
contra_ret = w_contra_shift * mkt['sprtrn'] + (1 - w_contra_shift) * mkt['rf']

err_df = pd.DataFrame({
    'date': mkt['date'],
    'port_ret': contra_ret,
    'sprtrn': mkt['sprtrn'],
    'weight': w_contra_shift,
    'S_t': mkt['S_t'],
    'S_neg_t': mkt['S_neg_t'],
})
test_err = err_df[err_df['date'] >= SPLIT_DATE].copy()
worst10 = test_err.nsmallest(10, 'port_ret')

print('=== Negativity_contrarian: 10 worst test-period days ===')
print(worst10.round(4).to_string(index=False))

print('\n--- Headlines on the worst day ---')
worst_day = worst10.iloc[0]['date']
sample = all_h_clu[all_h_clu['date'] == worst_day][['headline', 'pred_lr']].head(20)
print(f'Date: {worst_day.date()}')
for _, r in sample.iterrows():
    print(f'  [{r["pred_lr"]:>8s}]  {r["headline"]}')


## Section 17 — Transaction Cost Sensitivity

For each strategy, recompute Sharpe under per-trade costs of {0, 5, 10, 20} basis points. Cost on day *t* is `bps × |w_t − w_{t-1}|`.

In [ ]:
def sharpe_with_cost(weights, sprtrn, rf, cost_bps, mask):
    w_shift = weights.shift(1)
    gross = w_shift * sprtrn + (1 - w_shift) * rf
    turnover = (w_shift - w_shift.shift(1)).abs().fillna(0)
    cost = (cost_bps / 10000.0) * turnover
    net = gross - cost
    sub = net[mask].dropna()
    sub_rf = rf[mask].loc[sub.index]
    excess = sub - sub_rf
    ann_vol = sub.std() * np.sqrt(252)
    return (excess.mean() * 252) / ann_vol if ann_vol > 0 else np.nan


cost_levels = [0, 5, 10, 20]
cost_rows = []
for name, w_col in zip(strategy_names, weight_cols):
    for split, m in [('Train', mkt['date'] < SPLIT_DATE),
                     ('Test',  mkt['date'] >= SPLIT_DATE)]:
        row = {'strategy': name, 'split': split}
        for c in cost_levels:
            row[f'sharpe_{c}bps'] = sharpe_with_cost(
                mkt[w_col], mkt['sprtrn'], mkt['rf'], c, m.values
            )
        cost_rows.append(row)
cost_df = pd.DataFrame(cost_rows)
print(cost_df.round(3).to_string(index=False))

## Section 18 — Master Comparison Table

All strategies × both periods × all metrics: cumulative, annualised return, Sharpe, max DD, Calmar, FF3 α (annualised), α t-stat, α p-value, and Sharpe at 10 bps cost.

In [ ]:
master = results_df[display_cols].copy()
master = master.merge(
    ff3_df_out[['strategy', 'split', 'alpha_ann', 'tstat', 'pvalue']],
    on=['strategy', 'split'], how='left'
).rename(columns={'tstat': 'alpha_tstat', 'pvalue': 'alpha_pvalue'})

master = master.merge(
    cost_df[['strategy', 'split', 'sharpe_10bps']],
    on=['strategy', 'split'], how='left'
)

master = master.round(4)
print(master.to_string(index=False))

master.to_csv('master_comparison_clean.csv', index=False)
print('\nSaved master_comparison_clean.csv')

## Section 19 — Summary and Findings

**The hypothesis.** WSJ headline sentiment, aggregated daily, predicts
S&P 500 returns — and negative sentiment carries asymmetric (stronger)
predictive power than positive sentiment.

**The headline finding: the sign of the relationship inverts across
regimes.** In the quintile analysis (Section 10A.2), the highest-
negativity quintile (Q5) has mean next-day return of **+33% annualised on
the training period (2016–2021)** and **−60% annualised on the test
period (2022–2023)**. The "consistent" check prints `False`. This is the
single most important number in the notebook: in a bull market, high
negativity preceded a *bounce*; in a bear market, high negativity
preceded *more selling*. Any sentiment-trading rule that hard-codes one
direction will fail in the other regime.

**This regime flip is exactly what the contrarian/risk-off horse race
detects.** Both strategies use the same trigger — S_neg_t z-score >+1σ
above its 20-day mean — and take opposite actions. Out-of-sample:

- **negative_risk_off** posts test-period Sharpe **+0.68**, FF3 alpha
  **+12.0% annualised, t=2.39, p=0.017** (statistically significant
  positive alpha out-of-sample), and is the **only** sentiment strategy
  with positive Sharpe net of 10 bps transaction costs (+0.27).
- **negativity_contrarian** posts test-period Sharpe **−1.53**, FF3
  alpha **−13.6% annualised, t=−2.75, p=0.006** (statistically
  significant *negative* alpha — i.e., it loses systematically out-of-
  sample). Its training-period Sharpe was +0.79, so the failure is a
  pure regime-switch result, not a broken implementation.

The same trigger, opposite actions, opposite outcomes. The data picks
risk-off.

**Trend_momentum is a regime-conditional outperformer.** Training Sharpe
is 0.43 vs buy-and-hold's 0.87 — it sits in cash ~50% of the time during
the 2016–2021 bull market and underperforms. Test Sharpe is 0.76 vs
buy-and-hold's −0.06 — the same cash-defensive behaviour pays in the
2022 bear / 2023 recovery. FF3 alpha is +10.3% on test but not
significant (t=1.54, p=0.124). The interpretation is "trend-following
on improving sentiment is a defensive overlay, not a return enhancer".

**Surprise is dominated.** Test Sharpe 0.38, alpha +5.0% (p=0.43,
insignificant), 10bps Sharpe −0.58. Negative_risk_off beats it on
every metric. Surprise is the simpler "level above baseline" sibling of
trend_momentum and provides no additional information.

**Volatility prediction is real on training but fragile out-of-sample.**
S_neg_t IC vs |r_{t+1}| is +0.13 on train (p<0.001) but only +0.045 on
test (p=0.31). The earlier draft of this conclusion overstated the
robustness; with the test data in front of us, the claim "sentiment
predicts vol in both periods" does not hold. It predicts vol on training,
weakly and insignificantly on test.

**Limitations.**

- **Short test period** (24 months covering one bear market and one
  recovery). Sharpe ratios are unstable on samples this small, and the
  regime-switch result we identified is *itself* evidence that 24 months
  is not enough to characterise the relationship.
- **FinBERT circularity.** Labels we trained on were themselves model
  output. This biases training-period classifier metrics upward but does
  not affect test-period strategy returns.
- **Quantified regime instability.** The quintile sign flip is the
  cleanest evidence we have that the sentiment-return relationship is
  not stable across regimes. A naive single-regime trading rule will
  systematically fail when the regime turns.
- **Buy-and-hold has a small significant negative FF3 alpha** (−1.6%,
  p=0.045 on test). This is a benchmark composition artefact, not a
  finding about the SPX: `mktrf` is the broader CRSP value-weighted
  market, which marginally outperformed the SPX in this period.
  Strategy alphas should be read against the buy-and-hold alpha as a
  reference point, not against zero.
- **Daily aggregation discards intraday timing.** Headline timestamps
  are not used.

**Bottom line.** The asymmetry hypothesis is supported, but the *direction*
in which to trade it inverts across regimes. The only strategy in this
study with statistically significant positive out-of-sample alpha is
**negative_risk_off** (α=12% ann, p=0.017, positive at 10bps cost).
Trend_momentum is a useful defensive overlay in the bear regime but
does not produce significant alpha. The contrarian "buy-the-fear" rule
is significantly *negative* alpha out-of-sample — the asymmetry is real
but the bear-market reading of it ("step aside") wins decisively over
the bull-market reading ("buy the dip").
